# Repulsion Experiment On A Protein-Derived Theta Graph

This notebook is organized as a small workflow:

- load the project and external dependencies
- build the `1AOC` theta subgraph used in the experiment
- inspect the graph in two visualization styles
- run the repulsion-based geometric relaxation
- compute the Yamada polynomial on the relaxed graph
- plot the relaxed graph


In [1]:
import copy
import sys
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import networkx as nx
import numpy as np
import plotly.graph_objects as go
import requests
import sympy as sp
import torch
from Bio.PDB import PDBParser

repo_root = Path.cwd()
src_dir = repo_root / 'src'
if not src_dir.exists():
    raise RuntimeError('Run this notebook from the repository root so that ./src exists.')
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

import knotted_graph as kg

print(kg.__version__)
EXPORT_FIGS = False


0.1.3


## Build The Theta Graph

These helpers download the PDB file, extract the alpha-carbon backbone data, and construct the `theta_31` subgraph from chain `A` of `1AOC`.


In [2]:
# ============================================================
# Basic helpers
# ============================================================

def parse_pdb_chain_code(code):
    """
    Examples
    --------
    '5osqA' -> ('5OSQ', 'A')
    '1jyn'  -> ('1JYN', None)
    """
    code = code.strip()
    if len(code) < 4:
        raise ValueError("PDB code must have at least 4 characters.")

    pdb_id = code[:4].upper()
    chain_id = code[4:] if len(code) > 4 else None
    if chain_id == "":
        chain_id = None

    return pdb_id, chain_id


def download_pdb(pdb_id, out_path=None):
    pdb_id = pdb_id.upper()
    if out_path is None:
        out_path = f"{pdb_id}.pdb"

    url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
    r = requests.get(url)
    r.raise_for_status()

    with open(out_path, "w") as f:
        f.write(r.text)

    return out_path


# ============================================================
# Basic PDB / CA utilities
# ============================================================

def get_chain_ca_residues(pdb_path, chain_id, model_id=0):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("protein", pdb_path)
    model = list(structure)[model_id]
    chain = model[chain_id]

    residues = []
    for res in chain:
        hetflag, resseq, icode = res.id
        if hetflag.strip():
            continue
        if "CA" not in res:
            continue
        residues.append(res)
    return residues


def ca_coord(residues, curve_index):
    # paper uses C1, C2, ... as curve indices along the chain
    return np.asarray(residues[curve_index - 1]["CA"].coord, dtype=float)


def backbone_polyline(residues, a, b):
    step = 1 if b >= a else -1
    idxs = list(range(a, b + step, step))
    pts = np.array([ca_coord(residues, i) for i in idxs], dtype=float)
    return pts


def bridge_polyline(residues, a, b):
    # simplified plotting version:
    # draw bridge as a straight segment between the two CA positions
    pa = ca_coord(residues, a)
    pb = ca_coord(residues, b)
    return np.vstack([pa, pb])


def closure_polyline_via_point(residues, a, b, closure_point):
    pa = ca_coord(residues, a)
    pb = ca_coord(residues, b)
    return np.vstack([pa, closure_point, pb])


def concat_polylines(parts):
    out = []
    for p in parts:
        p = np.asarray(p, dtype=float)
        if len(out) == 0:
            out.append(p)
        else:
            # avoid duplicating the touching endpoint
            if np.allclose(out[-1][-1], p[0]):
                out.append(p[1:])
            else:
                out.append(p)
    return np.vstack(out)


# ============================================================
# Paper-like closure point
# ============================================================

def enclosing_sphere_point_from_chain(residues, seed=0, radius_scale=4):
    pts = np.array([res["CA"].coord for res in residues], dtype=float)
    center = pts.mean(axis=0)
    r = np.max(np.linalg.norm(pts - center, axis=1)) * radius_scale

    rng = np.random.default_rng(seed)
    v = rng.normal(size=3)
    v /= np.linalg.norm(v)
    return center + r * v


# ============================================================
# Build a theta-subgraph directly from the Table 2 arc specs
# ============================================================

def build_theta_subgraph_1aoc_theta31(pdb_path, chain_id="A", model_id=0, seed=0):
    """
    Figure-3 / Table-2 style construction for 1aocA, theta_31.

    Common endpoints: C140 and C134
    Arc1: C140 ... C161 $ C60 ... C65 $ C121 ... C134
    Arc2: C140 $ C88 ... C95 $ C10 ... C8 ⟺ C167 ... C172 $ C134
    Arc3: C140 ... C134
    """
    residues = get_chain_ca_residues(pdb_path, chain_id=chain_id, model_id=model_id)
    closure_pt = enclosing_sphere_point_from_chain(residues, seed=seed, radius_scale=0.1)

    # arc 1
    arc1 = concat_polylines([
        backbone_polyline(residues, 140, 161),
        bridge_polyline(residues, 161, 60),
        backbone_polyline(residues, 60, 65),
        bridge_polyline(residues, 65, 121),
        backbone_polyline(residues, 121, 134),
    ])

    # arc 2
    arc2 = concat_polylines([
        bridge_polyline(residues, 140, 88),
        backbone_polyline(residues, 88, 95),
        bridge_polyline(residues, 95, 10),
        backbone_polyline(residues, 10, 8),
        closure_polyline_via_point(residues, 8, 167, closure_pt),
        backbone_polyline(residues, 167, 172),
        bridge_polyline(residues, 172, 134),
    ])

    # arc 3
    arc3 = backbone_polyline(residues, 140, 134)

    G = nx.MultiGraph()

    u = ("theta_vertex", chain_id, 140)
    v = ("theta_vertex", chain_id, 134)

    G.add_node(u, pos=ca_coord(residues, 140), chain=chain_id, curve_index=140)
    G.add_node(v, pos=ca_coord(residues, 134), chain=chain_id, curve_index=134)

    G.add_edge(
        u, v,
        key="arc1",
        pts=arc1,
        arc_name="arc1",
        arc_spec="C140 ... C161 $ C60 ... C65 $ C121 ... C134",
        edge_type="theta_arc",
        color="red",
    )
    G.add_edge(
        u, v,
        key="arc2",
        pts=arc2,
        arc_name="arc2",
        arc_spec="C140 $ C88 ... C95 $ C10 ... C8 ⟺ C167 ... C172 $ C134",
        edge_type="theta_arc",
        color="blue",
    )
    G.add_edge(
        u, v,
        key="arc3",
        pts=arc3,
        arc_name="arc3",
        arc_spec="C140 ... C134",
        edge_type="theta_arc",
        color="green",
    )

    G.graph["theta_type_target"] = "theta_31"
    G.graph["pdb_chain"] = chain_id
    G.graph["closure_point"] = closure_pt
    return G


In [3]:
pdb_path = download_pdb("1AOC")
G_theta = build_theta_subgraph_1aoc_theta31(pdb_path, chain_id="A", seed=10)

print("nodes:", G_theta.number_of_nodes())
print("edges:", G_theta.number_of_edges())

for u, v, k, d in G_theta.edges(keys=True, data=True):
    print(k, d["arc_spec"], np.asarray(d["pts"]).shape)


kg.plot_3D_graph_plotly(G_theta)


nodes: 2
edges: 3
arc1 C140 ... C161 $ C60 ... C65 $ C121 ... C134 (42, 3)
arc2 C140 $ C88 ... C95 $ C10 ... C8 ⟺ C167 ... C172 $ C134 (20, 3)
arc3 C140 ... C134 (7, 3)


## Paper-Style Segment View

This optional view redraws the same theta graph with backbone, bridge, and closure segments separated visually.


In [4]:
def backbone_segment(residues, a, b, step_keep=3):
    step = 1 if b >= a else -1
    idxs = list(range(a, b + step, step))
    pts = np.array([ca_coord(residues, i) for i in idxs], dtype=float)

    # light simplification for plotting
    if len(pts) > 2 and step_keep > 1:
        keep = list(range(0, len(pts), step_keep))
        if keep[-1] != len(pts) - 1:
            keep.append(len(pts) - 1)
        pts = pts[keep]

    return {
        "kind": "backbone",
        "start": a,
        "end": b,
        "pts": pts,
    }


def bridge_segment(residues, a, b):
    return {
        "kind": "bridge",
        "start": a,
        "end": b,
        "pts": np.vstack([ca_coord(residues, a), ca_coord(residues, b)]),
    }


def closure_segment(residues, a, b, closure_point):
    return {
        "kind": "closure",
        "start": a,
        "end": b,
        "pts": np.vstack([ca_coord(residues, a), closure_point, ca_coord(residues, b)]),
    }


def concat_segments(segments):
    out = []
    for seg in segments:
        p = np.asarray(seg["pts"], dtype=float)
        if len(out) == 0:
            out.append(p)
        else:
            if np.allclose(out[-1][-1], p[0]):
                out.append(p[1:])
            else:
                out.append(p)
    return np.vstack(out)


def enclosing_sphere_point_from_pts(pts, radius_scale=2.5, seed=0):
    center = pts.mean(axis=0)
    r = np.max(np.linalg.norm(pts - center, axis=1)) * radius_scale
    rng = np.random.default_rng(seed)
    v = rng.normal(size=3)
    v /= np.linalg.norm(v)
    return center + r * v


# ============================================================
# Build 1aocA theta_31 with segment metadata
# ============================================================

def build_theta31_1aoc_segments(pdb_path, chain_id="A", model_id=0, seed=0):
    residues = get_chain_ca_residues(pdb_path, chain_id=chain_id, model_id=model_id)

    # closure point based on all CA points in chain
    chain_pts = np.array([res["CA"].coord for res in residues], dtype=float)
    closure_pt = enclosing_sphere_point_from_pts(chain_pts, radius_scale=0.1, seed=seed)

    # common branch endpoints
    a = 140
    b = 134

    arc1_segments = [
        backbone_segment(residues, 140, 161, step_keep=4),
        bridge_segment(residues, 161, 60),
        backbone_segment(residues, 60, 65, step_keep=2),
        bridge_segment(residues, 65, 121),
        backbone_segment(residues, 121, 134, step_keep=3),
    ]

    # if you want paper-like topology, this is the “bridge / closure / bridge” arc
    arc2_segments = [
        bridge_segment(residues, 140, 88),
        backbone_segment(residues, 88, 95, step_keep=2),
        bridge_segment(residues, 95, 10),
        backbone_segment(residues, 10, 8, step_keep=1),
        closure_segment(residues, 8, 167, closure_pt),
        backbone_segment(residues, 167, 172, step_keep=2),
        bridge_segment(residues, 172, 134),
    ]

    arc3_segments = [
        backbone_segment(residues, 140, 134, step_keep=3),
    ]

    G = nx.MultiGraph()

    u = ("theta_vertex", chain_id, a)
    v = ("theta_vertex", chain_id, b)

    G.add_node(u, pos=ca_coord(residues, a), curve_index=a)
    G.add_node(v, pos=ca_coord(residues, b), curve_index=b)

    G.add_edge(
        u, v, key="arc1",
        arc_name="arc1",
        segments=arc1_segments,
        pts=concat_segments(arc1_segments),
        arc_spec="C140 … C161 ↔ C60 … C65 ↔ C121 … C134",
    )
    G.add_edge(
        u, v, key="arc2",
        arc_name="arc2",
        segments=arc2_segments,
        pts=concat_segments(arc2_segments),
        arc_spec="C140 ↔ C88 … C95 ↔ C10 … C8 ⟺ C167 … C172 ↔ C134",
    )
    G.add_edge(
        u, v, key="arc3",
        arc_name="arc3",
        segments=arc3_segments,
        pts=concat_segments(arc3_segments),
        arc_spec="C140 … C134",
    )

    return G
def plot_theta_graph_paper_like(G):
    fig = go.Figure()

    arc_colors = {
        "arc1": "red",
        "arc2": "blue",
        "arc3": "green",
    }

    for _, _, key, data in G.edges(keys=True, data=True):
        arc_name = data["arc_name"]
        color = arc_colors.get(arc_name, "black")

        for seg in data["segments"]:
            pts = np.asarray(seg["pts"], dtype=float)
            kind = seg["kind"]

            if kind == "backbone":
                dash = "solid"
                width = 8
            elif kind == "bridge":
                dash = "dash"
                width = 6
            elif kind == "closure":
                dash = "dot"
                width = 5
            else:
                dash = "solid"
                width = 6

            fig.add_trace(go.Scatter3d(
                x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
                mode="lines",
                line=dict(color=color, width=width, dash=dash),
                name=f"{arc_name}-{kind}",
                showlegend=False,
            ))

    # show the two theta vertices clearly
    node_xyz = []
    node_text = []
    for n, d in G.nodes(data=True):
        p = np.asarray(d["pos"], dtype=float)
        node_xyz.append(p)
        node_text.append(str(n))

    node_xyz = np.asarray(node_xyz)
    fig.add_trace(go.Scatter3d(
        x=node_xyz[:, 0], y=node_xyz[:, 1], z=node_xyz[:, 2],
        mode="markers+text",
        marker=dict(size=8),
        text=["C140", "C134"],
        textposition="top center",
        name="theta vertices",
    ))

    fig.update_layout(
        scene=dict(
            xaxis_visible=False,
            yaxis_visible=False,
            zaxis_visible=False,
            aspectmode="data",
        ),
        margin=dict(l=0, r=0, b=0, t=30),
        title="1aocA θ-subgraph (paper-like plotting)",
    )
    return fig


In [5]:
G_theta_segments = build_theta31_1aoc_segments(pdb_path, chain_id="A", seed=7)
fig = plot_theta_graph_paper_like(G_theta_segments)
fig.show()


## Repulsion-Based Relaxation

The next cell defines the preprocessing and optimization helpers, and the run cell applies them to the theta graph built above.


In [6]:
# ============================================================
# Basic geometry helpers
# ============================================================

def _as_float_pts(pts):
    pts = np.asarray(pts, dtype=np.float64)
    if pts.ndim != 2 or pts.shape[1] not in (2, 3):
        raise ValueError(f"`pts` must have shape (N,2) or (N,3), got {pts.shape}")
    if pts.shape[1] == 2:
        pts = np.column_stack([pts, np.zeros(len(pts), dtype=np.float64)])
    return pts


def _as_float_pos(pos):
    pos = np.asarray(pos, dtype=np.float64)
    if pos.ndim != 1 or pos.shape[0] not in (2, 3):
        raise ValueError(f"`pos` must have shape (2,) or (3,), got {pos.shape}")
    if pos.shape[0] == 2:
        pos = np.r_[pos, 0.0]
    return pos


def _is_closed_polyline(pts: np.ndarray, tol: float = 1e-9) -> bool:
    pts = np.asarray(pts, dtype=np.float64)
    return len(pts) >= 3 and np.linalg.norm(pts[0] - pts[-1]) <= tol


def _curve_length_np(pts: np.ndarray, closed: bool) -> float:
    if len(pts) < 2:
        return 0.0
    if closed:
        diffs = np.roll(pts, -1, axis=0) - pts
        return float(np.linalg.norm(diffs, axis=1).sum())
    return float(np.linalg.norm(np.diff(pts, axis=0), axis=1).sum())


def densify_pts(pts, min_points=3):
    pts = _as_float_pts(pts)
    if pts.shape[0] >= min_points:
        return pts

    t_old = np.linspace(0.0, 1.0, pts.shape[0])
    t_new = np.linspace(0.0, 1.0, min_points)

    out = np.empty((min_points, 3), dtype=np.float64)
    for k in range(3):
        out[:, k] = np.interp(t_new, t_old, pts[:, k])
    return out


def _resample_polyline(
    pts: np.ndarray,
    spacing: Optional[float] = None,
    n: Optional[int] = None,
    closed: Optional[bool] = None,
) -> np.ndarray:
    pts = _as_float_pts(pts)
    if pts.shape[0] < 3:
        pts = densify_pts(pts, min_points=3)

    if closed is None:
        closed = _is_closed_polyline(pts)

    if closed and np.linalg.norm(pts[0] - pts[-1]) < 1e-12:
        pts_work = pts[:-1]
    else:
        pts_work = pts.copy()

    if closed:
        seg = np.roll(pts_work, -1, axis=0) - pts_work
        seglen = np.linalg.norm(seg, axis=1)
        total = seglen.sum()

        if total <= 0:
            out = densify_pts(pts_work, min_points=3)
            return np.vstack([out, out[:1]])

        s = np.concatenate([[0.0], np.cumsum(seglen)])
        base = np.vstack([pts_work, pts_work[:1]])

        if n is None:
            if spacing is None:
                n = max(3, len(pts_work))
            else:
                n = max(3, int(round(total / float(spacing))))

        sample_s = np.linspace(0.0, total, n, endpoint=False)
        out = np.empty((n, 3), dtype=np.float64)

        for k, val in enumerate(sample_s):
            idx = np.searchsorted(s, val, side="right") - 1
            idx = min(idx, len(seglen) - 1)
            ds = seglen[idx]
            t = 0.0 if ds == 0 else (val - s[idx]) / ds
            out[k] = (1.0 - t) * base[idx] + t * base[idx + 1]

        return np.vstack([out, out[:1]])

    seg = np.diff(pts_work, axis=0)
    seglen = np.linalg.norm(seg, axis=1)
    total = seglen.sum()

    if total <= 0:
        return densify_pts(pts_work, min_points=3)

    s = np.concatenate([[0.0], np.cumsum(seglen)])

    if n is None:
        if spacing is None:
            n = max(3, len(pts_work))
        else:
            n = max(3, int(round(total / float(spacing))) + 1)

    sample_s = np.linspace(0.0, total, n)
    out = np.empty((n, 3), dtype=np.float64)

    for k, val in enumerate(sample_s):
        idx = np.searchsorted(s, val, side="right") - 1
        idx = min(idx, len(seglen) - 1)
        ds = seglen[idx]
        t = 0.0 if ds == 0 else (val - s[idx]) / ds
        out[k] = (1.0 - t) * pts_work[idx] + t * pts_work[idx + 1]

    return out


# ============================================================
# Graph preprocessing
# ============================================================

def preprocess_graph_for_repel(G: nx.Graph, min_points: int = 3) -> nx.Graph:
    G2 = G.copy()

    # normalize node positions if present
    for n, nd in G2.nodes(data=True):
        if "pos" in nd:
            nd["pos"] = _as_float_pos(nd["pos"])

    # normalize edge polylines
    if G2.is_multigraph():
        for u, v, k, data in G2.edges(data=True, keys=True):
            if "pts" not in data:
                raise ValueError(f"Edge {(u, v, k)} is missing 'pts'")
            data["pts"] = densify_pts(data["pts"], min_points=min_points)
    else:
        for u, v, data in G2.edges(data=True):
            if "pts" not in data:
                raise ValueError(f"Edge {(u, v)} is missing 'pts'")
            data["pts"] = densify_pts(data["pts"], min_points=min_points)

    return G2


# ============================================================
# Packed curve structures
# ============================================================

@dataclass
class CurveRecord:
    edge_key: Tuple[Any, ...]
    u: Any
    v: Any
    closed: bool
    pts0: np.ndarray
    pin_start: bool
    pin_end: bool


@dataclass
class PackedCurves:
    X0: np.ndarray
    curve_ranges: List[Tuple[int, int]]
    closed_flags: np.ndarray
    pin_mask: np.ndarray
    target_lengths: np.ndarray
    endpoint_groups: Dict[Any, List[Tuple[int, str]]]


# ============================================================
# Convert graph to curves
# ============================================================

def nx_skeleton_to_curves(
    graph: nx.Graph,
    resample_spacing: Optional[float] = None,
    resample_n: Optional[int] = None,
    pin_special_vertices: bool = True,
    pin_endpoints_on_open_edges: bool = True,
) -> List[CurveRecord]:
    curves: List[CurveRecord] = []

    if graph.is_multigraph():
        edge_iter = ((u, v, k, data) for u, v, k, data in graph.edges(data=True, keys=True))
    else:
        edge_iter = ((u, v, None, data) for u, v, data in graph.edges(data=True))

    for u, v, k, data in edge_iter:
        pts = _as_float_pts(data["pts"])
        closed = (u == v) or _is_closed_polyline(pts)
        pts = _resample_polyline(pts, spacing=resample_spacing, n=resample_n, closed=closed)

        deg_u = graph.degree(u)
        deg_v = graph.degree(v)

        pin_start = False
        pin_end = False
        if not closed and pin_endpoints_on_open_edges:
            pin_start = True
            pin_end = True
            if pin_special_vertices:
                pin_start = (deg_u != 2)
                pin_end = (deg_v != 2)

        edge_key = (u, v, k) if k is not None else (u, v)
        curves.append(
            CurveRecord(
                edge_key=edge_key,
                u=u,
                v=v,
                closed=closed,
                pts0=pts,
                pin_start=pin_start,
                pin_end=pin_end,
            )
        )

    return curves


def pack_curves(curves: List[CurveRecord]) -> PackedCurves:
    xs = []
    curve_ranges = []
    closed_flags = []
    pin_mask = []
    target_lengths = []
    endpoint_groups: Dict[Any, List[Tuple[int, str]]] = {}

    start = 0
    for cid, rec in enumerate(curves):
        pts = rec.pts0.copy()

        if rec.closed and np.linalg.norm(pts[0] - pts[-1]) < 1e-12:
            pts = pts[:-1]

        n = len(pts)
        if n < 3:
            raise ValueError(f"Curve {cid} has too few points after preprocessing: {n}")

        xs.append(pts)
        curve_ranges.append((start, start + n))
        closed_flags.append(rec.closed)
        target_lengths.append(_curve_length_np(pts, rec.closed))

        pins = np.zeros(n, dtype=bool)
        if not rec.closed:
            pins[0] = rec.pin_start
            pins[-1] = rec.pin_end
            endpoint_groups.setdefault(rec.u, []).append((cid, "start"))
            endpoint_groups.setdefault(rec.v, []).append((cid, "end"))

        pin_mask.extend(pins.tolist())
        start += n

    return PackedCurves(
        X0=np.vstack(xs),
        curve_ranges=curve_ranges,
        closed_flags=np.asarray(closed_flags, dtype=bool),
        pin_mask=np.asarray(pin_mask, dtype=bool),
        target_lengths=np.asarray(target_lengths, dtype=np.float64),
        endpoint_groups=endpoint_groups,
    )


def curves_to_nx_graph(
    graph: nx.Graph,
    curves: List[CurveRecord],
    new_pts: List[np.ndarray],
) -> nx.Graph:
    G = copy.deepcopy(graph)

    for rec, pts in zip(curves, new_pts):
        pts = np.asarray(pts, dtype=np.float64)

        if G.is_multigraph():
            u, v, k = rec.edge_key
            G.edges[u, v, k]["pts"] = pts
        else:
            u, v = rec.edge_key
            G.edges[u, v]["pts"] = pts

        if not rec.closed:
            G.nodes[rec.u]["pos"] = pts[0].copy()
            G.nodes[rec.v]["pos"] = pts[-1].copy()

    return G


# ============================================================
# Constraint: shared node endpoints stay glued
# ============================================================

def _apply_endpoint_constraints_inplace(X: torch.Tensor, packed: PackedCurves):
    with torch.no_grad():
        for _, refs in packed.endpoint_groups.items():
            if len(refs) <= 1:
                continue

            coords = []
            indices = []
            for cid, side in refs:
                a, b = packed.curve_ranges[cid]
                idx = a if side == "start" else b - 1
                coords.append(X[idx])
                indices.append(idx)

            mean_pos = torch.stack(coords, dim=0).mean(dim=0)
            for idx in indices:
                X[idx] = mean_pos


# ============================================================
# Energies
# ============================================================

def _segment_primitives_torch(X: torch.Tensor, packed: PackedCurves):
    mids = []
    tans = []
    lens = []
    curve_ids = []
    local_seg_ids = []

    for cid, (a, b) in enumerate(packed.curve_ranges):
        Xi = X[a:b]

        if packed.closed_flags[cid]:
            X0 = Xi
            X1 = Xi.roll(-1, dims=0)
        else:
            X0 = Xi[:-1]
            X1 = Xi[1:]

        d = X1 - X0
        ell = torch.linalg.norm(d, dim=1).clamp_min(1e-12)
        t = d / ell[:, None]
        m = 0.5 * (X0 + X1)

        mids.append(m)
        tans.append(t)
        lens.append(ell)
        curve_ids.append(torch.full((len(ell),), cid, dtype=torch.long, device=X.device))
        local_seg_ids.append(torch.arange(len(ell), dtype=torch.long, device=X.device))

    return (
        torch.cat(mids, dim=0),
        torch.cat(tans, dim=0),
        torch.cat(lens, dim=0),
        torch.cat(curve_ids, dim=0),
        torch.cat(local_seg_ids, dim=0),
    )


def segment_tpe_energy_torch(
    X: torch.Tensor,
    packed: PackedCurves,
    alpha: float = 2.0,
    beta: float = 4.0,
    local_exclusion: int = 1,
    eps: float = 1e-12,
):
    mids, tans, lens, curve_id, local_seg_id = _segment_primitives_torch(X, packed)

    d = mids[None, :, :] - mids[:, None, :]
    r2 = (d * d).sum(dim=-1).clamp_min(eps)
    r = torch.sqrt(r2)

    Ti = tans[:, None, :]
    Tj = tans[None, :, :]

    di_par = (d * Ti).sum(dim=-1)
    dj_par = ((-d) * Tj).sum(dim=-1)

    di_perp2 = (r2 - di_par * di_par).clamp_min(eps)
    dj_perp2 = (r2 - dj_par * dj_par).clamp_min(eps)

    di_perp = torch.sqrt(di_perp2)
    dj_perp = torch.sqrt(dj_perp2)

    Eij = (di_perp**alpha + dj_perp**alpha) / (r**beta)
    Eij = Eij * (lens[:, None] * lens[None, :])

    nseg = mids.shape[0]
    mask = torch.ones((nseg, nseg), dtype=torch.bool, device=X.device)
    idx = torch.arange(nseg, device=X.device)
    mask[idx, idx] = False

    same_curve = curve_id[:, None] == curve_id[None, :]
    near_local = torch.abs(local_seg_id[:, None] - local_seg_id[None, :]) <= local_exclusion
    mask = mask & ~(same_curve & near_local)

    # closed curves: first and last segment are adjacent
    for cid, _ in enumerate(packed.curve_ranges):
        if packed.closed_flags[cid]:
            seg_ids = torch.where(curve_id == cid)[0]
            if len(seg_ids) > 1:
                mask[seg_ids[0], seg_ids[-1]] = False
                mask[seg_ids[-1], seg_ids[0]] = False

    return torch.triu(Eij * mask, diagonal=1).sum()


def curve_lengths_torch(X: torch.Tensor, packed: PackedCurves) -> torch.Tensor:
    out = []
    for cid, (a, b) in enumerate(packed.curve_ranges):
        Xi = X[a:b]
        if packed.closed_flags[cid]:
            Li = torch.linalg.norm(Xi.roll(-1, dims=0) - Xi, dim=1).sum()
        else:
            Li = torch.linalg.norm(Xi[1:] - Xi[:-1], dim=1).sum()
        out.append(Li)
    return torch.stack(out)


def bending_energy_torch(X: torch.Tensor, packed: PackedCurves) -> torch.Tensor:
    E = X.new_tensor(0.0)
    for cid, (a, b) in enumerate(packed.curve_ranges):
        Xi = X[a:b]
        if packed.closed_flags[cid]:
            E = E + ((Xi.roll(-1, 0) - 2.0 * Xi + Xi.roll(1, 0)) ** 2).sum()
        else:
            if len(Xi) > 2:
                E = E + ((Xi[2:] - 2.0 * Xi[1:-1] + Xi[:-2]) ** 2).sum()
    return E


# ============================================================
# Main optimizer
# ============================================================

def repel_skeleton_graph(
    graph: nx.Graph,
    *,
    resample_spacing: Optional[float] = 2.0,
    resample_n: Optional[int] = None,
    alpha: float = 2.0,
    beta: float = 4.0,
    repulsion_weight: float = 1.0,
    length_weight: float = 5,
    bending_weight: float = 1e-2,
    local_exclusion: int = 1,
    step_size: float = 1e-3,
    iterations: int = 200,
    pin_special_vertices: bool = True,
    pin_endpoints_on_open_edges: bool = True,
    device: str = "cpu",
    verbose: bool = True,
    grad_clip: float = 1.0,
):
    graph = preprocess_graph_for_repel(graph, min_points=3)

    curves = nx_skeleton_to_curves(
        graph,
        resample_spacing=resample_spacing,
        resample_n=resample_n,
        pin_special_vertices=pin_special_vertices,
        pin_endpoints_on_open_edges=pin_endpoints_on_open_edges,
    )
    packed = pack_curves(curves)

    X0 = torch.tensor(packed.X0, dtype=torch.float64, device=device)
    X = X0.clone().detach().requires_grad_(True)

    pin_mask = torch.as_tensor(packed.pin_mask, dtype=torch.bool, device=device)
    target_lengths = torch.tensor(packed.target_lengths, dtype=torch.float64, device=device)

    opt = torch.optim.Adam([X], lr=step_size)

    history = []
    best_E = None
    best_X = None

    for it in range(iterations):
        opt.zero_grad()

        E_rep = segment_tpe_energy_torch(
            X, packed,
            alpha=alpha,
            beta=beta,
            local_exclusion=local_exclusion,
        )
        lengths = curve_lengths_torch(X, packed)
        E_len = ((lengths - target_lengths) ** 2).sum()
        E_bend = bending_energy_torch(X, packed)

        E = repulsion_weight * E_rep + length_weight * E_len + bending_weight * E_bend
        E.backward()

        with torch.no_grad():
            if X.grad is not None:
                X.grad[pin_mask] = 0.0
                gnorm = torch.linalg.norm(X.grad)
                if torch.isfinite(gnorm) and gnorm > grad_clip:
                    X.grad *= grad_clip / (gnorm + 1e-12)

        opt.step()

        with torch.no_grad():
            X[pin_mask] = X0[pin_mask]
            _apply_endpoint_constraints_inplace(X, packed)

        val = float(E.detach().cpu())
        hist = {
            "iter": it,
            "E_total": val,
            "E_rep": float(E_rep.detach().cpu()),
            "E_len": float(E_len.detach().cpu()),
            "E_bend": float(E_bend.detach().cpu()),
        }
        history.append(hist)

        if best_E is None or val < best_E:
            best_E = val
            best_X = X.detach().clone()

        if verbose and (it % max(1, iterations // 10) == 0 or it == iterations - 1):
            print(
                f"[{it:4d}] "
                f"E={hist['E_total']:.6e}  "
                f"rep={hist['E_rep']:.6e}  "
                f"len={hist['E_len']:.6e}  "
                f"bend={hist['E_bend']:.6e}"
            )

    X_final = best_X.detach().cpu().numpy()

    new_pts = []
    for cid, rec in enumerate(curves):
        a, b = packed.curve_ranges[cid]
        pts = X_final[a:b].copy()
        if rec.closed:
            pts = np.vstack([pts, pts[:1]])
        new_pts.append(pts)

    G_new = curves_to_nx_graph(graph, curves, new_pts)
    return G_new, history


# ============================================================
# Self-contained wrapper with the SAME style of call
# ============================================================

def run_repulsive_curves_on_nx_graph(
    G_closed: nx.Graph,
    *,
    rcurves_exe: Optional[str] = None,   # ignored in this pure-python version
    alpha: float = 2.0,
    beta: float = 4.0,
    repulsion_weight: float = 1.0,
    iteration_limit: int = 200,
    fix_special_vertices: bool = True,
    fix_endpoint_vertices: bool = False,   # accepted for compatibility
    fix_special_tangents: bool = False,    # accepted for compatibility
    fix_barycenter: bool = True,
    fix_edgelengths: bool = True,
    keep_files: bool = False,              # accepted for compatibility
    timeout: Optional[int] = None,         # accepted for compatibility
    workdir: Optional[str] = None,         # accepted for compatibility
    resample_spacing: float = 2.0,
    step_size: float = 1e-3,
    length_weight: Optional[float] = None,
    bending_weight: float = 1e-2,
    device: str = "cpu",
    verbose: bool = True,
):
    """
    Pure-Python replacement for the earlier external-wrapper function.
    It accepts the same kind of call and returns (G_rep, info).
    """
    t0 = time.time()

    G_in = preprocess_graph_for_repel(G_closed, min_points=3)

    # optional recentering
    if fix_barycenter:
        all_pts = []
        for _, nd in G_in.nodes(data=True):
            if "pos" in nd:
                all_pts.append(_as_float_pos(nd["pos"]))
        if all_pts:
            c = np.mean(np.vstack(all_pts), axis=0)
            for _, nd in G_in.nodes(data=True):
                if "pos" in nd:
                    nd["pos"] = _as_float_pos(nd["pos"]) - c
            if G_in.is_multigraph():
                for _, _, _, data in G_in.edges(data=True, keys=True):
                    data["pts"] = _as_float_pts(data["pts"]) - c
            else:
                for _, _, data in G_in.edges(data=True):
                    data["pts"] = _as_float_pts(data["pts"]) - c

    # map compatibility flags to our penalties
    if length_weight is None:
        length_weight = 5.0 if fix_edgelengths else 0.5

    pin_special = bool(fix_special_vertices)
    pin_open_endpoints = bool(fix_endpoint_vertices) or bool(fix_special_vertices)

    G_rep, hist = repel_skeleton_graph(
        G_in,
        resample_spacing=resample_spacing,
        alpha=alpha,
        beta=beta,
        iterations=iteration_limit,
        step_size=step_size,
        length_weight=length_weight,
        bending_weight=bending_weight,
        repulsion_weight=repulsion_weight,
        pin_special_vertices=pin_special,
        pin_endpoints_on_open_edges=pin_open_endpoints,
        device=device,
        verbose=verbose,
    )

    info = {
        "backend": "python_segment_repulsion",
        "elapsed_seconds": time.time() - t0,
        "iterations": iteration_limit,
        "alpha": alpha,
        "beta": beta,
        "repulsion_weight": repulsion_weight,
        "resample_spacing": resample_spacing,
        "step_size": step_size,
        "length_weight": length_weight,
        "bending_weight": bending_weight,
        "history": hist,
        "rcurves_exe_used": None,
        "note": "Pure Python fallback; no external executable used.",
    }
    return G_rep, info


In [7]:
G_rep, info = run_repulsive_curves_on_nx_graph(
    G_theta,
    alpha=2,
    beta=4,
    repulsion_weight=4.0,
    iteration_limit=5000,
    fix_special_vertices=True,
    fix_special_tangents=False,
    fix_barycenter=False,
    fix_edgelengths=False,
    resample_spacing=6.0,
    step_size=0.1,
    device="cpu",
    verbose=True,
)

print(info["backend"], info["elapsed_seconds"])


[   0] E=8.338580e+02  rep=2.064215e+02  len=2.019484e-28  bend=8.171951e+02
[ 500] E=1.655982e+02  rep=3.653738e+01  len=1.860101e+00  bend=1.851864e+03
[1000] E=1.517027e+02  rep=3.221126e+01  len=2.483683e+00  bend=2.161586e+03
[1500] E=1.479083e+02  rep=3.113539e+01  len=2.883807e+00  bend=2.192482e+03
[2000] E=1.452326e+02  rep=3.023658e+01  len=4.065237e+00  bend=2.225367e+03
[2500] E=1.266479e+02  rep=2.618101e+01  len=1.667558e+00  bend=2.109005e+03
[3000] E=1.238479e+02  rep=2.551203e+01  len=1.654603e+00  bend=2.097245e+03
[3500] E=1.230215e+02  rep=2.539597e+01  len=1.691603e+00  bend=2.059176e+03
[4000] E=1.225416e+02  rep=2.535325e+01  len=1.754610e+00  bend=2.025128e+03
[4500] E=1.218288e+02  rep=2.517964e+01  len=1.715699e+00  bend=2.025243e+03
[4999] E=1.216371e+02  rep=2.513448e+01  len=1.654283e+00  bend=2.027201e+03
python_segment_repulsion 4.858551263809204


## Yamada Polynomial Of The Relaxed Graph

This computes the Yamada polynomial on the repelled graph `G_rep`.


In [8]:
A = sp.symbols('A')

kg.compute_yamada_safely(
    skeleton_graph=G_rep,
    variable=A,
)


-A**4 - A**3 - 2*A**2 - A - 1

## Plot The Relaxed Graph


In [9]:
kg.plot_3D_graph_plotly(G_rep)
